# Trustworthy Explainable Search — Full Analysis Pipeline (Cleaned)

This notebook implements the complete computational pipeline described in
Section 3 (Methodology) of the term paper *"Trustworthy Explainable Search:
Detecting and Visualizing Hallucinated Explanations in Multimodal Cultural
Heritage Collections,"* from explanation generation through the six
qualitative case studies.

It merges and cleans up two source notebooks: one that generated the 460
explanations with `Qwen2.5-VL-7B-Instruct`, and one that ran the
downstream consistency/reliability analysis. Removed along the way:
GitHub push cells, five near-duplicate "batch N" generation cells
(consolidated into one resumable loop), duplicate metadata-inspection
cells, a stray file-existence checker, a `_FIXED`-suffix patch cell, and
two conflicting claim-classification functions (kept one).

**Two things fixed here that are worth double-checking against your own
reported numbers:**

1. **Reliability score (§3.8).** An earlier version of the downstream
   notebook computed the reliability score as
   `0.80 × NLI consistency + 0.20 × metadata support` for any image with
   at least one metadata-checkable claim. The paper instead describes the
   score as NLI consistency alone. This notebook implements the paper's
   version directly, and keeps the weighted blend only inside a
   **sensitivity-analysis** cell (Section 12), which shows the choice of
   weighting barely matters given how sparse metadata coverage is.
2. **Sentence-pair double-counting (§3.4).** The original alignment cell
   compared every sentence to every other sentence in *both* directions
   (`(A, B)` and `(B, A)` as two separate rows), then only deduplicated
   pairs *after* the similarity threshold was applied. This notebook
   instead generates each unordered pair exactly once with
   `itertools.combinations`, so the "eligible pairs" and
   "candidate pairs" counts are correct from the start rather than
   needing a later dedup step.

If you already have a `documerica_image_reliability_scores.csv` or
`documerica_sentence_alignment.csv` from a previous run, it's worth
diffing their totals against a fresh run of this notebook.

## 1. Setup and archival metadata (§3.1)

Loads the Documerica metadata spreadsheet (96 records) and downloads the
corresponding images. Four downloads are expected to fail with HTTP 403
(§3.1) — these are logged, not silently dropped, so the gap between the
96-record dataset and the 92-image computational sample stays visible.

In [ ]:
import os
import gc
import time
import requests
import pandas as pd
import numpy as np
import torch

from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image

PROJECT_DIR = Path("./documerica_project")
IMAGE_DIR = PROJECT_DIR / "images"
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"

for d in (IMAGE_DIR, OUTPUT_DIR, CHECKPOINT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR.resolve())

In [ ]:
METADATA_PATH = "documerica_images_cleaned.xlsx"
metadata_df = pd.read_excel(METADATA_PATH)

print("Metadata records:", len(metadata_df))
metadata_df.head()

In [ ]:
download_log = []

for _, row in tqdm(metadata_df.iterrows(), total=len(metadata_df), desc="Downloading images"):
    image_name = str(row["image_name"])
    if not Path(image_name).suffix:
        image_name += ".jpg"
    output_path = IMAGE_DIR / image_name

    try:
        response = requests.get(
            row["image_url"], timeout=30, headers={"User-Agent": "Mozilla/5.0"}
        )
        response.raise_for_status()
        output_path.write_bytes(response.content)
        download_log.append({"image_name": image_name, "image_id": row["image_id"], "status": "success"})
    except Exception as e:
        download_log.append({
            "image_name": image_name, "image_id": row["image_id"],
            "status": "failed", "error": str(e),
        })
    time.sleep(0.1)

download_log_df = pd.DataFrame(download_log)
download_log_df.to_csv(OUTPUT_DIR / "image_download_log.csv", index=False)

print("Successful downloads:", (download_log_df["status"] == "success").sum())
print("Failed downloads:    ", (download_log_df["status"] == "failed").sum())

## 2. Explanation generation with Qwen2.5-VL-7B-Instruct (§3.2, Appendix A)

Five explanations are generated per successfully downloaded image (460
total), using the restrictive prompt in Appendix A and a fixed sampling
configuration (temperature 0.8, top-p 0.9) with five different seeds.

**Note on runtime.** Generating 460 explanations with a 7B vision-language
model is slow even on a GPU, and free-tier notebook platforms (Kaggle,
Colab) enforce session time limits. The loop below writes to a checkpoint
CSV after every single explanation and skips anything already present, so
it is always safe to stop and re-run this cell across multiple sessions —
which is what the original analysis did in practice, split across five
manually-triggered runs. Consolidating that into one resumable loop here
removes the need for near-duplicate "batch 1 / batch 2 / ..." cells.

In [ ]:
!pip install -q -U transformers accelerate qwen-vl-utils

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

gc.collect()
torch.cuda.empty_cache()

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto", low_cpu_mem_usage=True,
)
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model.eval()

print("Model loaded:", MODEL_NAME, "| device:", model.device)

In [ ]:
PROMPT = """
You are analyzing a historical cultural heritage photograph.

Describe and explain the image based ONLY on what is visually observable.
Separate direct visual observations from interpretations or inferences.

Use cautious language such as "appears to be", "may be", or
"could indicate" whenever something cannot be established directly
from the image.

Discuss relevant:
- people and their visible actions,
- objects and structures,
- setting and environment,
- activities,
- visible text or signs,
- possible contextual clues.

Do NOT invent or confidently assert:
- dates,
- locations,
- names,
- occupations,
- historical events,
- cultural identities,
- relationships,
- photographer information,
- object identities that are visually uncertain,
- or other facts that cannot be established from the image.

If something cannot be determined from the image, explicitly state that
it cannot be determined.

Produce a concise but informative explanation of approximately
120-180 words.
"""

SEEDS = [1001, 1002, 1003, 1004, 1005]

In [ ]:
def generate_explanation(image_path, seed, temperature=0.8, top_p=0.9, max_new_tokens=120):
    """Generate one explanation for one image at one sampling seed."""
    gc.collect()
    torch.cuda.empty_cache()

    image = Image.open(image_path).convert("RGB")
    image.thumbnail((512, 512), Image.Resampling.LANCZOS)  # smaller image -> faster inference

    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": PROMPT}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt")
    inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            top_p=top_p, do_sample=True, use_cache=True,
        )

    trimmed = [out[len(inp):] for inp, out in zip(inputs["input_ids"], generated_ids)]
    output_text = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

    del inputs, generated_ids, trimmed
    gc.collect()
    torch.cuda.empty_cache()
    return output_text.strip()

In [ ]:
# One row per (image, seed). Re-running this cell resumes from the checkpoint
# instead of regenerating explanations that already exist.
CHECKPOINT_FILE = CHECKPOINT_DIR / "all_explanations_checkpoint.csv"
CHECKPOINT_COLUMNS = ["image_name", "image_id", "explanation_number", "seed", "explanation"]

if CHECKPOINT_FILE.exists():
    generation_results = pd.read_csv(CHECKPOINT_FILE)
else:
    generation_results = pd.DataFrame(columns=CHECKPOINT_COLUMNS)

available_images = download_log_df[download_log_df["status"] == "success"]
print(f"Resuming with {len(generation_results)} explanations already generated "
      f"out of {len(available_images) * len(SEEDS)} targeted.")

In [ ]:
# NOTE: on a time-limited GPU session, re-run this cell as many times as
# needed -- already-completed (image, seed) pairs are skipped automatically.
for _, row in tqdm(available_images.iterrows(), total=len(available_images), desc="Images"):
    image_path = IMAGE_DIR / row["image_name"]

    for explanation_number, seed in enumerate(SEEDS, start=1):
        already_done = (
            (generation_results["image_name"] == row["image_name"])
            & (generation_results["explanation_number"] == explanation_number)
        ).any()
        if already_done:
            continue

        try:
            explanation = generate_explanation(image_path, seed=seed)
        except Exception as e:
            print(f"Error on {row['image_name']} (explanation {explanation_number}): {e}")
            continue

        new_row = {
            "image_name": row["image_name"], "image_id": row["image_id"],
            "explanation_number": explanation_number, "seed": seed, "explanation": explanation,
        }
        generation_results = pd.concat([generation_results, pd.DataFrame([new_row])], ignore_index=True)
        generation_results.to_csv(CHECKPOINT_FILE, index=False)  # save after every explanation

print("Total explanations so far:", len(generation_results))
print("Images covered:", generation_results["image_name"].nunique())

In [ ]:
# Final, de-duplicated explanations file (460 rows: 92 images x 5 explanations).
EXPLANATIONS_CSV = OUTPUT_DIR / "documerica_all_460_explanations.csv"

df = (
    generation_results
    .drop_duplicates(subset=["image_name", "explanation_number"])
    .sort_values(["image_name", "explanation_number"])
    .reset_index(drop=True)
)
df.to_csv(EXPLANATIONS_CSV, index=False, quoting=1)

TEXT_COL = "explanation"
print("Final explanations:", len(df))
print("Unique images:     ", df["image_name"].nunique())
print("\nExplanations per image:")
print(df.groupby("image_name").size().value_counts().sort_index())

## 3. Sentence segmentation (§3.3)

Each explanation is split into sentences with NLTK's Punkt tokenizer. The
sentence (not the whole explanation) is the unit compared across
explanations, so that local disagreements are not hidden by whole-document
similarity.

In [ ]:
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import sent_tokenize

sentence_rows = []
for idx, row in df.iterrows():
    for sent_num, sentence in enumerate(sent_tokenize(str(row[TEXT_COL])), start=1):
        sentence_rows.append({
            "image_name": row["image_name"],
            "explanation_index": idx,
            "sentence_number": sent_num,
            "sentence": sentence,
        })

sentences_df = pd.DataFrame(sentence_rows).reset_index(drop=True)
sentences_df["sentence_id"] = sentences_df.index

print("Total sentences:", len(sentences_df))
print("Unique images:  ", sentences_df["image_name"].nunique())
print("Sentences per explanation (mean):", len(sentences_df) / len(df))

sentences_df.to_csv(OUTPUT_DIR / "documerica_sentences.csv", index=False)
sentences_df.head()

## 4. Sentence embeddings (§3.4)

Each sentence is embedded with `all-MiniLM-L6-v2` (Reimers & Gurevych,
2019), producing 384-dimensional vectors compared with cosine similarity.

In [ ]:
!pip install -q -U sentence-transformers

In [ ]:
EMBEDDINGS_PATH = OUTPUT_DIR / "sentence_embeddings.npy"

if EMBEDDINGS_PATH.exists():
    embeddings = np.load(EMBEDDINGS_PATH)
    print("Loaded cached embeddings:", embeddings.shape)
else:
    from sentence_transformers import SentenceTransformer

    embed_device = "cuda" if torch.cuda.is_available() else "cpu"
    embedder = SentenceTransformer("all-MiniLM-L6-v2", device=embed_device)
    embeddings = embedder.encode(
        sentences_df["sentence"].tolist(),
        batch_size=32, show_progress_bar=True, normalize_embeddings=True,
    )
    np.save(EMBEDDINGS_PATH, embeddings)
    print("Computed and cached embeddings:", embeddings.shape)

assert embeddings.shape[0] == len(sentences_df), "Embedding count does not match sentence count."
embeddings_normalized = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
print("Embedding dimension:", embeddings.shape[1])

## 5. Sentence-pair generation and cosine similarity (§3.4)

For each image, every pair of sentences drawn from *different*
explanations is compared exactly once (pairs from the same explanation are
excluded, and each unordered pair is generated a single time via
`itertools.combinations` — see the note at the top of this notebook about
why that matters).

In [ ]:
from itertools import combinations

pair_rows = []
for image_name, group in sentences_df.groupby("image_name"):
    records = group.to_dict("records")
    for a, b in combinations(records, 2):
        if a["explanation_index"] == b["explanation_index"]:
            continue  # same explanation -> not an independent comparison
        pair_rows.append({
            "image_name": image_name,
            "source_id": a["sentence_id"], "target_id": b["sentence_id"],
            "source_sentence": a["sentence"], "target_sentence": b["sentence"],
        })

pairs_df = pd.DataFrame(pair_rows)
source_vecs = embeddings_normalized[pairs_df["source_id"].to_numpy()]
target_vecs = embeddings_normalized[pairs_df["target_id"].to_numpy()]
pairs_df["similarity"] = np.sum(source_vecs * target_vecs, axis=1)

print("Eligible unordered sentence pairs:", len(pairs_df))
pairs_df.to_csv(OUTPUT_DIR / "documerica_sentence_alignment.csv", index=False)
pairs_df.head()

In [ ]:
# Similarity-threshold sensitivity (Table 2 / Figure 3)
THRESHOLDS = [0.50, 0.60, 0.70, 0.80]
similarity_sensitivity_df = pd.DataFrame([
    {"similarity_threshold": t, "candidate_pairs": (pairs_df["similarity"] >= t).sum()}
    for t in THRESHOLDS
])
similarity_sensitivity_df.to_csv(OUTPUT_DIR / "similarity_threshold_sensitivity.csv", index=False)
similarity_sensitivity_df

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(similarity_sensitivity_df["similarity_threshold"],
         similarity_sensitivity_df["candidate_pairs"], marker="o")
plt.xlabel("Cosine similarity threshold")
plt.ylabel("Candidate sentence pairs")
plt.title("Sensitivity to Cosine Similarity Threshold")
plt.xticks(THRESHOLDS)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "similarity_threshold_sensitivity.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
SIMILARITY_THRESHOLD = 0.60  # primary threshold used for the NLI stage (§3.4)
candidate_pairs_df = pairs_df[pairs_df["similarity"] >= SIMILARITY_THRESHOLD].reset_index(drop=True)
print("Candidate pairs passed to NLI:", len(candidate_pairs_df))

## 6. Natural language inference (§3.5)

Each candidate pair is classified as entailment / neutral / contradiction
with `cross-encoder/nli-deberta-v3-base` (He et al., 2021).

In [ ]:
from transformers import pipeline

NLI_MODEL = "cross-encoder/nli-deberta-v3-base"
nli_device = 0 if torch.cuda.is_available() else -1

nli_pipeline = pipeline("text-classification", model=NLI_MODEL, tokenizer=NLI_MODEL,
                         device=nli_device, truncation=True)
nli_tokenizer = nli_pipeline.tokenizer
nli_model = nli_pipeline.model
nli_model.eval()

torch_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nli_model.to(torch_device)
print("NLI model loaded. Labels:", nli_model.config.id2label)

In [ ]:
from tqdm.auto import tqdm

BATCH_SIZE = 16
nli_results = []

for start in tqdm(range(0, len(candidate_pairs_df), BATCH_SIZE), desc="Running NLI"):
    batch = candidate_pairs_df.iloc[start:start + BATCH_SIZE]

    inputs = nli_tokenizer(
        batch["source_sentence"].astype(str).tolist(),
        batch["target_sentence"].astype(str).tolist(),
        padding=True, truncation=True, max_length=512, return_tensors="pt",
    )
    inputs = {k: v.to(torch_device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = nli_model(**inputs).logits
    probabilities = torch.softmax(logits, dim=-1)
    predicted_ids = torch.argmax(probabilities, dim=-1)

    for j in range(len(batch)):
        row = batch.iloc[j]
        label_id = predicted_ids[j].item()
        nli_results.append({
            "image_name": row["image_name"], "source_id": row["source_id"], "target_id": row["target_id"],
            "source_sentence": row["source_sentence"], "target_sentence": row["target_sentence"],
            "similarity": row["similarity"],
            "nli_label": nli_model.config.id2label[label_id],
            "nli_score": probabilities[j, label_id].item(),
        })

nli_results_df = pd.DataFrame(nli_results)
nli_results_df.to_csv(OUTPUT_DIR / "documerica_nli_results.csv", index=False)
print("\nNLI label distribution:")
print(nli_results_df["nli_label"].value_counts())

## 7. Image-level consistency and contradiction rate (§3.5, Table 4 / Figure 1)

For each image: `consistency = entailment / (entailment + contradiction)`,
with neutral pairs excluded from the denominator. Contradiction rate is
`contradiction / total pairs`.

In [ ]:
def summarize_image(group):
    entailment = (group["nli_label"] == "entailment").sum()
    contradiction = (group["nli_label"] == "contradiction").sum()
    neutral = (group["nli_label"] == "neutral").sum()
    denom = entailment + contradiction
    return pd.Series({
        "nli_pairs": len(group), "entailment_count": entailment,
        "contradiction_count": contradiction, "neutral_count": neutral,
        "contradiction_rate": contradiction / len(group),
        "consistency_score": entailment / denom if denom > 0 else np.nan,
    })

image_nli_summary = (
    nli_results_df.groupby("image_name").apply(summarize_image, include_groups=False).reset_index()
)
image_nli_summary["consistency_percent"] = image_nli_summary["consistency_score"] * 100
image_nli_summary.to_csv(OUTPUT_DIR / "documerica_image_nli_summary.csv", index=False)

print("Mean contradiction rate:  ", (image_nli_summary["contradiction_rate"] * 100).mean().round(2), "%")
print("Median contradiction rate:", (image_nli_summary["contradiction_rate"] * 100).median().round(2), "%")
image_nli_summary.sort_values("contradiction_rate", ascending=False).head(10)

In [ ]:
def contradiction_bucket(rate):
    pct = rate * 100
    if pct == 0: return "0%"
    if pct <= 5: return ">0-5%"
    if pct <= 10: return ">5-10%"
    if pct <= 20: return ">10-20%"
    return ">20%"

image_nli_summary["contradiction_bucket"] = image_nli_summary["contradiction_rate"].apply(contradiction_bucket)
bucket_order = ["0%", ">0-5%", ">5-10%", ">10-20%", ">20%"]
bucket_counts = image_nli_summary["contradiction_bucket"].value_counts().reindex(bucket_order, fill_value=0)
print(bucket_counts.to_frame("images"))

In [ ]:
plot_df = image_nli_summary.sort_values("contradiction_rate", ascending=False).reset_index(drop=True)

plt.figure(figsize=(14, 6))
plt.bar(plot_df["image_name"], plot_df["contradiction_rate"] * 100)
plt.xlabel("Image"); plt.ylabel("Contradiction rate (%)")
plt.title("Contradiction Rate Across Digital Documerica Images")
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "contradiction_rate_by_image.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Claim clustering (§3.6)

Sentences within the same image are greedily clustered by cosine
similarity into candidate "claims": a sentence joins the first existing
cluster whose representative sentence is similar enough, otherwise it
starts a new cluster.

In [ ]:
def cluster_image_sentences(indices, threshold):
    '''Greedy single-pass clustering of one image's sentences.
    Returns a dict: sentence_id -> cluster_id (local to this image).'''
    representative_ids, assignment = [], {}
    for idx in indices:
        vector = embeddings_normalized[idx]
        cluster_id = None
        for c_id, rep_idx in enumerate(representative_ids):
            if np.dot(vector, embeddings_normalized[rep_idx]) >= threshold:
                cluster_id = c_id
                break
        if cluster_id is None:
            representative_ids.append(idx)
            cluster_id = len(representative_ids) - 1
        assignment[idx] = cluster_id
    return assignment

In [ ]:
# Clustering-threshold sensitivity
CLUSTER_THRESHOLDS = [0.70, 0.75, 0.80]
cluster_sensitivity_rows = []
for threshold in CLUSTER_THRESHOLDS:
    sizes = []
    for image_name, group in sentences_df.groupby("image_name"):
        assignment = cluster_image_sentences(group["sentence_id"].tolist(), threshold)
        sizes.extend(pd.Series(assignment.values()).value_counts().tolist())
    cluster_sensitivity_rows.append({
        "clustering_threshold": threshold, "total_clusters": len(sizes),
        "mean_cluster_size": np.mean(sizes), "median_cluster_size": np.median(sizes),
        "max_cluster_size": np.max(sizes),
    })
claim_clustering_sensitivity_df = pd.DataFrame(cluster_sensitivity_rows)
claim_clustering_sensitivity_df.to_csv(OUTPUT_DIR / "claim_clustering_sensitivity.csv", index=False)
claim_clustering_sensitivity_df

In [ ]:
# Final clustering at the primary threshold (0.75)
CLAIM_THRESHOLD = 0.75
claim_records = []
for image_name, group in sentences_df.groupby("image_name"):
    assignment = cluster_image_sentences(group["sentence_id"].tolist(), CLAIM_THRESHOLD)
    for _, row in group.iterrows():
        claim_records.append({
            "image_name": image_name, "sentence_id": row["sentence_id"],
            "explanation_index": row["explanation_index"], "sentence_number": row["sentence_number"],
            "sentence": row["sentence"], "claim_id": f"{image_name}_claim_{assignment[row['sentence_id']]}",
        })

claim_level_df = pd.DataFrame(claim_records)
claim_level_df.to_csv(OUTPUT_DIR / "documerica_claim_level_dataset.csv", index=False)
print("Total claim clusters:", claim_level_df["claim_id"].nunique())
claim_level_df.head()

## 9. Match claims to archival metadata (§3.7)

Matches each claim cluster to the row of archival metadata for its image.

In [ ]:
import re

def strip_extension(name):
    return re.sub(r"\.[^.]+$", "", str(name).strip())

claim_level_df["image_key"] = claim_level_df["image_name"].apply(strip_extension)
metadata_df["image_key"] = metadata_df["image_name"].apply(strip_extension)

claim_metadata_df = claim_level_df.merge(
    metadata_df, on="image_key", how="left", suffixes=("_claim", "_metadata"), validate="many_to_one",
)
print(f"Claims matched to metadata: {claim_metadata_df['image_id'].notna().mean() * 100:.2f}%")

In [ ]:
claim_level_metadata = (
    claim_metadata_df.groupby("claim_id", as_index=False)
    .agg({
        "image_key": "first", "image_name_claim": "first",
        "sentence": lambda s: " ".join(dict.fromkeys(str(x) for x in s)),
        "city": "first", "state": "first", "date_taken": "first",
        "keywords": "first", "exhibit_description": "first", "byline": "first",
    })
    .rename(columns={"image_name_claim": "image_name", "sentence": "claim_text"})
)
for col in ["city", "state", "date_taken", "keywords", "exhibit_description", "byline"]:
    claim_level_metadata[col] = claim_level_metadata[col].fillna("").astype(str).str.strip()

print("Candidate claim clusters:", len(claim_level_metadata))
claim_level_metadata.head()

## 10. Claim-field classification (§3.6)

Each claim is assigned a preliminary field — `date`, `location`,
`context`, or `visual` (the fallback) — via keyword matching. This is the
single, canonical classifier: the same one used for both the reported
field-distribution statistics and for deciding which claims are
metadata-checkable in the next section.

In [ ]:
DATE_WORDS = ["date", "dated", "year", "century", "decade",
              "1960", "1970", "1980", "1990", "2000", "historical", "modern", "early", "late"]
LOCATION_WORDS = ["location", "located", "city", "state", "street", "road", "avenue",
                  "town", "building", "site", "neighborhood", "district", "phoenix", "arizona"]
CONTEXT_WORDS = ["government", "official", "epa", "environmental", "historical", "industrial",
                 "residential", "commercial", "institution", "facility", "community",
                 "cultural", "event", "work"]

def classify_claim_field(text):
    text = str(text).lower()
    if any(w in text for w in DATE_WORDS): return "date"
    if any(w in text for w in LOCATION_WORDS): return "location"
    if any(w in text for w in CONTEXT_WORDS): return "context"
    return "visual"

claim_level_metadata["claim_field"] = claim_level_metadata["claim_text"].apply(classify_claim_field)
field_counts = claim_level_metadata["claim_field"].value_counts()
pd.DataFrame({"count": field_counts, "percent": (field_counts / len(claim_level_metadata) * 100).round(2)})

## 11. Metadata support categories (§3.7, Table 3)

Each claim is checked against its matched metadata and assigned one of
four categories. Visual claims are always `not_metadata_checkable`, since
archival metadata generally cannot verify purely visual detail.

In [ ]:
def assess_metadata_support(row):
    claim = row["claim_text"].lower()
    field = row["claim_field"]
    city, state, date = row["city"].lower(), row["state"].lower(), row["date_taken"].lower()
    keywords, description = row["keywords"].lower(), row["exhibit_description"].lower()

    if field == "visual":
        return "not_metadata_checkable"
    if field == "location":
        candidates = [v for v in (city, state) if v]
        if not candidates: return "not_metadata_checkable"
        return "potentially_supported" if any(v in claim for v in candidates) else "unverifiable"
    if field == "date":
        if not date: return "not_metadata_checkable"
        claim_years = set(re.findall(r"\b(?:18|19|20)\d{2}\b", claim))
        metadata_years = set(re.findall(r"\b(?:18|19|20)\d{2}\b", date))
        if not claim_years or not metadata_years: return "unverifiable"
        return "potentially_supported" if claim_years & metadata_years else "not_supported_by_metadata"
    if field == "context":
        metadata_text = " ".join(v for v in (keywords, description) if v)
        if not metadata_text: return "not_metadata_checkable"
        claim_words = set(re.findall(r"\b[a-z]{4,}\b", claim))
        metadata_words = set(re.findall(r"\b[a-z]{4,}\b", metadata_text))
        return "potentially_supported" if claim_words & metadata_words else "unverifiable"
    return "unverifiable"

claim_level_metadata["metadata_category"] = claim_level_metadata.apply(assess_metadata_support, axis=1)
claim_level_metadata.to_csv(OUTPUT_DIR / "documerica_claim_metadata_validation.csv", index=False)

category_counts = claim_level_metadata["metadata_category"].value_counts()
pd.DataFrame({"count": category_counts, "percent": (category_counts / len(claim_level_metadata) * 100).round(2)})

## 12. Image-level reliability indicator (§3.8, Figure 2)

The primary reliability score is **NLI consistency alone** (matches the
paper). The weighted blend is kept only as a sensitivity check.

In [ ]:
reliability_df = image_nli_summary.copy()
reliability_df["reliability_score"] = reliability_df["consistency_percent"]

def reliability_category(score):
    if score < 60: return "low"
    if score < 80: return "moderate"
    return "high"

reliability_df["reliability_category"] = reliability_df["reliability_score"].apply(reliability_category)
reliability_df = reliability_df.sort_values("reliability_score").reset_index(drop=True)
reliability_df.to_csv(OUTPUT_DIR / "documerica_image_reliability_scores.csv", index=False)

print(reliability_df["reliability_score"].describe().round(2))
print("\n", reliability_df["reliability_category"].value_counts().reindex(["low", "moderate", "high"]))

In [ ]:
plot_df = reliability_df.sort_values("reliability_score").reset_index(drop=True)

plt.figure(figsize=(14, 6))
plt.plot(range(len(plot_df)), plot_df["reliability_score"], marker="o", markersize=3, linewidth=1)
plt.axhline(60, linestyle="--", linewidth=1); plt.axhline(80, linestyle="--", linewidth=1)
plt.xlabel("Images ordered by reliability score"); plt.ylabel("Reliability score")
plt.title("Image-Level Reliability Scores for Documerica Explanations")
plt.text(len(plot_df) - 1, 60, "  Low / Moderate threshold (60)", va="bottom")
plt.text(len(plot_df) - 1, 80, "  Moderate / High threshold (80)", va="bottom")
plt.ylim(0, 105); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "image_reliability_scores.png", dpi=300, bbox_inches="tight")
plt.show()

### 12a. Verifying that no metadata weighting was applied

A quick, mechanical check that the reliability score really is pure NLI
consistency and not a blend: `reliability_score` should equal
`consistency_percent` for **every** image, not just on average. This is
also the check referenced in the paper's Section 3.8 — the reported
dataset-wide minimum reliability score is identical to the raw consistency
score of whichever single image attains that minimum, which would only
happen by coincidence under a genuine metadata-weighted blend.

In [ ]:
max_difference = (reliability_df["reliability_score"] - reliability_df["consistency_percent"]).abs().max()
print("Max |reliability_score - consistency_percent| across all images:", max_difference)
assert max_difference < 1e-9, "reliability_score is not pure NLI consistency -- check for a metadata blend."

lowest = reliability_df.loc[reliability_df["reliability_score"].idxmin()]
print(f"\nLowest-reliability image: {lowest['image_name']}")
print(f"  reliability_score:   {lowest['reliability_score']:.2f}")
print(f"  consistency_percent: {lowest['consistency_percent']:.2f}")
print("These two values match exactly, confirming no metadata weighting was blended in.")

In [ ]:
# Sensitivity check: does a metadata weighting change the ranking?
from scipy.stats import spearmanr

metadata_by_image = (
    claim_level_metadata.groupby("image_name")["metadata_category"]
    .agg(supported=lambda s: (s == "potentially_supported").sum(),
         unsupported=lambda s: (s == "not_supported_by_metadata").sum())
)
metadata_by_image["checkable"] = metadata_by_image["supported"] + metadata_by_image["unsupported"]
metadata_by_image["support_score"] = np.where(
    metadata_by_image["checkable"] > 0,
    metadata_by_image["supported"] / metadata_by_image["checkable"], np.nan,
)

sens_df = reliability_df.merge(metadata_by_image[["support_score"]], on="image_name", how="left")
primary_scores = sens_df["reliability_score"]

rows = []
for nli_w, meta_w in [(1.00, 0.00), (0.90, 0.10), (0.80, 0.20), (0.70, 0.30)]:
    scores = np.where(
        sens_df["support_score"].notna(),
        (nli_w * sens_df["consistency_score"] + meta_w * sens_df["support_score"]) * 100,
        sens_df["consistency_score"] * 100,
    )
    rho, p = spearmanr(primary_scores, scores)
    rows.append({"nli_weight": nli_w, "metadata_weight": meta_w,
                 "mean_score": np.mean(scores), "spearman_rho_vs_primary": rho, "p_value": p})

weight_sensitivity_df = pd.DataFrame(rows)
weight_sensitivity_df.to_csv(OUTPUT_DIR / "reliability_weight_sensitivity.csv", index=False)
weight_sensitivity_df.round(4)

## 13. Manual NLI validation (§3.9, §4.4)

The 30-pair manual validation (10 genuine / 18 false positives / 2
ambiguous) was done by hand — there is no code for the judging itself.
This cell only exports a random sample of contradiction pairs for review.

In [ ]:
SAMPLE_SIZE, RANDOM_SEED = 30, 42

contradiction_pairs = nli_results_df[nli_results_df["nli_label"] == "contradiction"]
n_sample = min(SAMPLE_SIZE, len(contradiction_pairs))
manual_review_sample = contradiction_pairs.sample(n=n_sample, random_state=RANDOM_SEED).copy()
manual_review_sample["manual_judgment"] = ""  # fill in by hand

manual_review_sample.to_csv(OUTPUT_DIR / "manual_nli_review_sample.csv", index=False)
print(f"Exported {len(manual_review_sample)} contradiction pairs for manual review.")
manual_review_sample[["image_name", "source_sentence", "target_sentence", "nli_score", "manual_judgment"]].head()

## 14. Case studies (§3.9, §4.7)

Extracts explanations and contradiction pairs for the six images discussed
as qualitative case studies: three low-consistency (0071, 0039, 0030) and
three high-consistency (0063, 0018, 0074).

In [ ]:
SELECTED_CASES = ["documerica_0071.jpg", "documerica_0039.jpg", "documerica_0030.jpg",
                  "documerica_0063.jpg", "documerica_0018.jpg", "documerica_0074.jpg"]

case_explanations = df[df["image_name"].isin(SELECTED_CASES)].sort_values("image_name").reset_index(drop=True)
case_explanations.to_csv(OUTPUT_DIR / "case_study_explanations.csv", index=False)

case_contradictions = (
    nli_results_df[nli_results_df["image_name"].isin(SELECTED_CASES) & (nli_results_df["nli_label"] == "contradiction")]
    .sort_values(["image_name", "nli_score"], ascending=[True, False]).reset_index(drop=True)
)
case_contradictions.to_csv(OUTPUT_DIR / "case_study_contradictions.csv", index=False)

print("Case-study explanations:", len(case_explanations))
print("Case-study contradiction pairs:", len(case_contradictions))
case_contradictions.head()

In [ ]:
case_summary = (
    reliability_df[reliability_df["image_name"].isin(SELECTED_CASES)]
    [["image_name", "reliability_score", "reliability_category", "consistency_score", "contradiction_rate", "nli_pairs"]]
    .merge(case_contradictions.groupby("image_name").size().rename("contradiction_pairs"), on="image_name", how="left")
)
case_summary["contradiction_pairs"] = case_summary["contradiction_pairs"].fillna(0).astype(int)
case_summary = case_summary.sort_values("reliability_score").reset_index(drop=True)
case_summary.to_csv(OUTPUT_DIR / "case_study_summary.csv", index=False)
case_summary.round(3)